<img src="logo.png" alt="Vegeta" width="240">

# MERLIN — the design (26)

MERLIN is a compact electric fixed-wing that leaves a launch stand, **dashes** to a reported hotspot, flies crosswind
passes through the smoke column with a forward gas sensor (CO, CO₂, particles), and comes home for a belly landing — rapid
sampling of a plume, a smoke column or a gas release, and **ignition confirmation**: is the reported hotspot a real fire,
and how big is it.

**This notebook designs the aircraft** — in three variants that differ only in the propulsor (a nose **ducted fan** whose
jet leaves as a ring around the fuselage, a **tractor** or a **pusher** propeller) — and **exports the design** to
`designs/data/merlin_design.json`. The flights come after, in their own notebooks, which read that file instead of designing
again: `26b_merlin_mission` (the race to the fire, the mountain, the smoke mission, the movie) and `27_merlin_race` (can any
propulsor design win).

Nothing is designed twice: the airframe is notebook 09's `FixedWing` (its wing, tail and sections; `designs/merlin.py`
subclasses it) and the nose duct is notebook 25b's fan housing (`designs/ducted_fan.py`).

```
1  mission and mass budget → 2  the three MERLINs in CAD → 3  size, mass and drag build-up
4  wing FEA (pull-up, dash gust), two revisions → 5  whole-aircraft RANS per variant (optional) → 6  print the nose unit
7  export the design
```

The OpenFOAM cells run when you run them (`VEGETA_SKIP_OPENFOAM=1` skips them; the drag build-up is the polar then).

In [ ]:
# DEBUG: test the notebook end to end before the full run. On: FEA and CFD are mocked (vegeta.mock: fixed numbers,
# seconds instead of hours, the result cache off). Turn it on here or from the shell:
#   VEGETA_DEBUG=1 jupyter nbconvert --to notebook --execute <notebook>.ipynb --output <notebook>-run.ipynb
DEBUG = False
from vegeta import mock
DEBUG = mock.setup(DEBUG)

In [ ]:
import json, math, os, shutil, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Video, display
from vegeta import dedalus, talos, aeromant, mellonia, core, boreas
from vegeta.dedalus import viz as dviz
from vegeta.talos import viz as tviz
from vegeta.aeromant import viz as aviz
from vegeta.mellonia.examples import GENERIC_PLA_0_2MM
sys.path.insert(0, "designs")
import merlin as mdesign
import merlin_flight as mf

ROOT = Path("_runs/merlin"); shutil.rmtree(ROOT, ignore_errors=True); ROOT.mkdir(parents=True)
OUT = Path("output") / "26_merlin"; OUT.mkdir(parents=True, exist_ok=True)       # every movie of this notebook
RHO, NU, G = 1.225, 1.5e-5, 9.81
RUN_CFD = os.environ.get("VEGETA_SKIP_OPENFOAM") != "1"
QUICK = os.environ.get("MERLIN_QUICK") == "1"
KINDS = ("edf", "tractor", "pusher")
NAME = {"edf": "ducted fan", "tractor": "tractor propeller", "pusher": "pusher propeller"}
COLOR = {"edf": "#d62728", "tractor": "#1f77b4", "pusher": "#ff7f0e"}   # the ducted fan in red
print(f"CFD {'on' if RUN_CFD else 'skipped'}; {'QUICK check' if QUICK else 'full tables'}")

# Part 1 — the airframe

## 1. Mission and mass budget — edit here

The three MERLINs share everything except the propulsor: the same printed airframe, the same 6S 8000 mAh battery, the
same flight controller and the same gas sensor. Each propulsor brings its own mass: the EDF unit (its motor inside,
320 g) with an 80 A controller, or a 10-inch propeller with an outrunner and a 60 A controller. **One electrical power
limit for all**: 1900 W, the EDF's catalogue full power (about 12 C from this pack) — so the race compares propulsors,
not batteries or motors.

In [ ]:
MISSION = mf.Mission(sampling_agl_m=150.0, sampling_s=180.0, launch_speed=12.0, battery_wh=6 * 3.7 * 8.0,
                     usable_fraction=0.85, reserve_fraction=0.15)
DISTANCES_KM = mf.DISTANCES_KM                        # 5, 8, 10, 20, 30 km
MAX_ELECTRICAL_W, DRIVE_EFF = 1900.0, 0.85
COMMON = dict(mf.COMMON_MASS_KG)                      # kg; the airframe line is replaced from the CAD below
UNIT_MASS = dict(mf.UNIT_MASS_KG)
SHELL_AREAL_DENSITY_G_MM2 = 0.7e-3                    # 0.7 kg/m^2: ~1 mm LW-PLA shell + spar + glue (assumption;
                                                      # 09a writes 1.1 kg/m^2 but uses 1.1e-4 g/mm^2 = 0.11 kg/m^2)
pd.DataFrame({"mass [g]": {**{k: v * 1000 for k, v in COMMON.items()}, **{f"propulsion: {NAME[k]}": v * 1000 for k, v in UNIT_MASS.items()}}})

## 2. The three MERLINs (Dedalus)

`Merlin` is `FixedWing` with a new fuselage and a `propulsion` switch: `edf` puts notebook 25's duct (bell-mouth,
shroud, area-ruled nozzle, seven stator vanes) on the nose and lets the fuselage run through the nozzle, so the jet
leaves as a ring around it; `tractor` puts the motor and spinner on the nose; `pusher` gives it a clean sensor nose and
puts the motor at the end of the tail cone, the propeller behind the tail. The design file is copied into the run folder,
as in 09a, so a copilot proposal edits the copy.

In [ ]:
design_file = ROOT / "merlin.py"
shutil.copy(Path("designs/merlin.py"), design_file)
merlin = dedalus.load_design(f"{design_file}:Merlin")
display(pd.DataFrame(merlin.params.table()).set_index("name"))
GEOM = {k: merlin.generate(propulsion=k) for k in KINDS}
for k, g in GEOM.items():
    print(f"{NAME[k]:<18s} valid {g.shape.isValid()}, {len(g.shape.Solids())} solid, bounding box {np.round(g.dimensions, 0)} mm")

In [ ]:
try:
    for k in KINDS:
        dviz.show(dviz.plot3d(GEOM[k]))
except ImportError as e:
    print("3D view needs pyvista:", e)
fig, ax = plt.subplots(3, 1, figsize=(13, 7.5), sharex=True)
for a, k in zip(ax, KINDS):
    for poly in mdesign.outline({"propulsion": k})["side"]:
        a.fill(poly[:, 0], poly[:, 1], color=COLOR[k], alpha=0.35, ec="k", lw=0.6)
    a.axvline(mdesign.Merlin.layout(merlin.resolve(propulsion=k))["prop_x"] / 1000, color="k", ls=":", lw=1)
    a.set_aspect("equal"); a.set_title(f"{NAME[k]}: side view (propulsor plane dotted)", fontsize=10); a.grid(alpha=0.3)
ax[-1].set_xlabel("x [m]")
fig.tight_layout()

In [ ]:
fig = dviz.plot_sections(GEOM["edf"], normal="x", positions=[-215.0, -190.0, -150.0, 100.0], cols=4)   # lip, stators, nozzle exit, wing

## 3. Size, mass and drag build-up

The airframe mass comes from the CAD's surface area at 09a's shell density. The parasite drag is a component
build-up (`merlin.drag_buildup`: flat-plate friction on each component's own length times its form factor, plus 10 %
interference) with an allowance for the skid, the sensor intake and the antennas. The EDF nacelle's outside friction
and its jet scrubbing the fuselage are charged to the fan (part 2), the propellers' slipstream to the propellers (the
thrust deduction).

In [ ]:
V_DESIGN = 25.0
p0 = merlin.resolve()
area = GEOM["pusher"].surface_area
COMMON["airframe (printed shells)"] = round(area * SHELL_AREAL_DENSITY_G_MM2 / 1000, 3)
rows = {}
for k in KINDS:
    b = mdesign.drag_buildup({"propulsion": k}, V_DESIGN, NU)
    mass = sum(COMMON.values()) + UNIT_MASS[k]
    S = b["planform"]
    rows[NAME[k]] = {"all-up mass [kg]": mass, "wing area [m²]": S, "aspect ratio": b["aspect_ratio"],
                     "wing loading [N/m²]": mass * G / S, "stall speed (Cl_max 1.2) [m/s]": math.sqrt(2 * mass * G / (RHO * S * 1.2)),
                     "Cd0 build-up": b["cd0"], "Cd0 with allowance": (b["cd_area_m2"] + mf.EXTRA_CD_AREA_M2) / S,
                     **{f"Cd·A {c} [cm²]": v * 1e4 for c, v in b["parts_m2"].items()}}
SIZE = pd.DataFrame(rows)
print(f"airframe from the CAD's surface area: {COMMON['airframe (printed shells)'] * 1000:.0f} g")
SIZE.round(4)

## 4. Wing structure (Talos)

The same model as 09a (solid-equivalent LW-PLA, the centre section clamped, the lift as a pressure on the lower
skins), with MERLIN's load cases. The dash is the new one: at 50–65 m/s a gust hits much harder than at a mapping
drone's cruise. Pratt's gust load factor `Δn = K_g ρ V U a S / (2 W)` (a 7.5 m/s gust, the alleviation
`K_g = 0.88 μ / (5.3 + μ)` with the mass ratio `μ = 2 (W/S) / (ρ c a g)`) at 60 m/s sets it — a light, fast aircraft
feels gusts hard.

| case | loads | represents |
|---|---|---|
| `pull_up` | 3 × W on the lower skins | the turns between the sampling passes and the recovery from a gust upset |
| `dash_gust` | (1 + Δn) × W | a vertical gust at the dash speed |

In [ ]:
LW_PLA = talos.Material("LW-PLA printed wing (solid-equivalent)", youngs_modulus=400.0, poissons_ratio=0.35,
                        density=0.6e-9, yield_strength=12.0, source="assumed (09a); replace with coupon tests")
W_MAX = max(SIZE.loc["all-up mass [kg]"]) * G
S_W = SIZE.loc["wing area [m²]"].iloc[0]
V_GUST, U_GUST, A_LIFT = 60.0, 7.5, 2 * math.pi * 0.85 * 6.5 / (6.5 + 2)
MU = 2 * W_MAX / S_W / (RHO * p0["root_chord"] * (1 + p0["taper"]) / 2000 * A_LIFT * G)
K_G = 0.88 * MU / (5.3 + MU)
N_GUST = 1 + K_G * RHO * V_GUST * U_GUST * A_LIFT * S_W / (2 * W_MAX)
print(f"gust load factor at {V_GUST:.0f} m/s: {N_GUST:.1f} (mass ratio {MU:.1f}, alleviation {K_G:.2f})")

def lower_skins(step):
    info = talos.inspect_step(step, units="mm-N-MPa")
    skins = [s for s in info.surfaces if s.kind == "BSpline surface" and s.area > 5e4]
    return [min((s for s in skins if (s.centroid[1] > 0) == right), key=lambda s: s.centroid[2]).tag for right in (False, True)]

def wing_model(rev, n, name):
    yc = rev.params["fuselage_diameter"] / 2 + 5.0
    regions = [talos.SurfacesInBox("root", (-1.0, -yc - 0.1, -100.0, 400.0, yc + 0.1, 100.0)), talos.Surfaces("lift", lower_skins(rev.step))]
    return talos.StructuralModel(rev.step, "mm-N-MPa", LW_PLA, regions, [talos.FixedSupport("root")],
                                 [talos.Pressure("lift", n * W_MAX / (S_W * 1e6))], talos.MeshSettings(element_size=10.0), name=name)

def pull_up(rev):
    return wing_model(rev, 3.0, "pull_up")

def dash_gust(rev):
    return wing_model(rev, N_GUST, "dash_gust")

ws = core.Workspace.create(ROOT / "workspace", name="MERLIN")
design = ws.add_design("merlin", f"{design_file}:Merlin")
w1 = design.new_revision(part="wing", note="baseline wing NACA 2412")
w1.generate()
w2 = w1.branch(thickness=0.15, note="NACA 2415: thicker wing")
w2.generate()
FEA = {}                 # the evaluations as run: they carry the solver results (an evaluation read back from disk has only its record)
for rev in (w1, w2):
    for case in (pull_up, dash_gust):
        FEA[(rev.id, case.__name__)] = rev.run_fea(case.__name__, case, progress=True)

def row(rev):
    d = {"note": rev.record.get("note", ""), "thickness": rev.params["thickness"]}
    for case in ("pull_up", "dash_gust"):
        ev = rev.evaluation("fea", case)
        d[f"{case} tip [mm]"] = ev.metrics.get("max_displacement", np.nan) if ev and ev.ok else np.nan
        d[f"{case} SF"] = ev.metrics.get("safety_factor_yield", np.nan) if ev and ev.ok else np.nan
    return d

WING = pd.DataFrame({r.id: row(r) for r in (w1, w2)}).T
display(WING)
if WING["dash_gust SF"].notna().any():
    preferred = ws.revision(WING["dash_gust SF"].astype(float).idxmax())
    preferred.label("preferred", note="higher gust safety factor")
else:
    preferred = w1
    print("FEA not run (CalculiX missing?): the baseline wing stays; see ws.status()")
ws.status()

In [ ]:
ev = FEA.get((preferred.id, "dash_gust"))
if ev is not None and ev.ok and ev.tool_results:
    res = ev.tool_results[-1]
    tviz.show(tviz.plot_results(res, field="von_mises"))
    fig = talos.plot_along_axis(res, axis="y", quantity="displacement", component=2)
else:
    print("no dash_gust result to plot (the FEA did not run here: is CalculiX installed?)")

## 5. Whole-aircraft aerodynamics (Aeromant, optional)

One coarse RANS run per variant at 2° (the dash flies at a low lift coefficient) — 09a's case with
`reference_length = span/4`. A run replaces that variant's build-up Cd0 by the CFD point through a parabolic polar
(Oswald 0.8). Without OpenFOAM the build-up stays — section 3's numbers.

In [ ]:
V_CFD, AOA = 30.0, 2.0
CFD_POLAR = {}

def cruise(rev, workdir):
    return aeromant.CFDCase("rans_ksst_external", rev.stl,
        dict(velocity=V_CFD, kinematic_viscosity=NU, density=RHO, reference_area=S_W, reference_length=p0["span"] / 4000,
             center_of_rotation=(0.07, 0.0, 0.0), iterations=400, residual_target=1e-4, surface_level=4, near_level=3,
             wake_level=2, cells_per_length=2.0),
        workdir=workdir, geometry_units="mm", environment=aeromant.OpenFOAMEnvironment.detect())

if RUN_CFD:
    for k in KINDS:
        rev = preferred.branch(part="aircraft", propulsion=k, angle_of_attack_deg=AOA, note=f"{NAME[k]}, {AOA} deg, CFD")
        rev.generate(stl_tolerance=0.2)
        ev = rev.run_cfd("cruise_2deg", cruise, progress=True)
        if ev.ok:
            cl, cd = ev.metrics["Cl"], ev.metrics["Cd"]
            CFD_POLAR[k] = cd - cl ** 2 / (math.pi * SIZE.loc["aspect ratio", NAME[k]] * 0.8)
            print(f"{NAME[k]}: Cl {cl:.3f}, Cd {cd:.4f} -> Cd0 {CFD_POLAR[k]:.4f} (build-up {SIZE.loc['Cd0 with allowance', NAME[k]]:.4f})")
else:
    print("CFD skipped (VEGETA_SKIP_OPENFOAM=1): the drag build-up is the polar")

## 6. Print the nose unit (Mellonia)

The EDF's duct with its stator vanes and the ring they stand on, printed upright on the nozzle exit's flat face.

In [ ]:
nose = design.new_revision(part="nose", propulsion="edf", note="EDF duct with stators, for printing")
nose.generate()
ev_p = nose.run_print("exit_down", GENERIC_PLA_0_2MM, mellonia.Orientation(rotate_y=90))
print(ev_p)

## 7. Export the design

What the flights need, as plain numbers (`designs/data/merlin_design.json`; `merlin_flight.load_design` reads it): the design
file's parameters with the preferred wing, the common masses (the airframe from the CAD) and the propulsion units' at 1900 W,
the drag allowance, the correction the CFD makes to each variant's build-up (zero where it did not run), the polar's Cl_max
and Oswald factor, the size and drag table, the wing's FEA, and the mission settings.

In [ ]:
import dataclasses
pp = merlin.resolve()
MERLIN_PARAMS = {k: v for k, v in pp.items() if k not in ("part", "propulsion", "angle_of_attack_deg") and not k.startswith("edf_")}
MERLIN_PARAMS["thickness"] = preferred.params["thickness"]
CD0_CORRECTION = {k: (CFD_POLAR[k] - mdesign.drag_buildup(dict(MERLIN_PARAMS, propulsion=k), V_DESIGN, NU)["cd0"]) if k in CFD_POLAR else 0.0
                  for k in KINDS}

def plain(x):
    return None if isinstance(x, float) and not np.isfinite(x) else x

DESIGN = {"source": "26_merlin", "merlin_params": MERLIN_PARAMS, "common_mass_kg": COMMON, "unit_mass_kg_1900W": UNIT_MASS,
          "extra_cd_area_m2": mf.EXTRA_CD_AREA_M2, "cd0_correction": CD0_CORRECTION, "cd0_cfd": CFD_POLAR, "cl_max": 1.2, "oswald": 0.8,
          "variants": {c: {k: plain(float(v)) for k, v in SIZE[c].items()} for c in SIZE.columns},
          "wing_fea": {r: {k: plain(v) for k, v in WING.loc[r].items()} for r in WING.index},
          "preferred_wing": {"revision": preferred.id, "thickness": preferred.params["thickness"]},
          "gust": {"speed_m_s": V_GUST, "gust_m_s": U_GUST, "load_factor": N_GUST},
          "mission": dataclasses.asdict(MISSION), "distances_km": list(DISTANCES_KM),
          "max_electrical_w": MAX_ELECTRICAL_W, "drive_efficiency": DRIVE_EFF}
mf.DESIGN_JSON.parent.mkdir(parents=True, exist_ok=True)
mf.DESIGN_JSON.write_text(json.dumps(DESIGN, indent=1, default=float))
print("written", mf.DESIGN_JSON, "| preferred wing thickness", MERLIN_PARAMS["thickness"], "| CFD corrections", CD0_CORRECTION)